In [2]:
import pandas as pd
from pymongo import MongoClient
from ollama import chat
import json

In [6]:
def read_collection() -> pd.DataFrame:
    """
    Reads all records from the configured MongoDB collection and returns them as a pandas DataFrame.
    The MongoDB connection details: database name and collection name are read from the config.json file.

    @returns:
        pd.DataFrame: Pandas DataFrame containing all records retrieved from the MongoDB collection.
    """
    # Reading configurations file to extract database name and collection names
    with open("config.json", "r") as file:
        config_data = json.load(file)

    # Establishing connection with MongoDB & reading customer reviews
    client = MongoClient(config_data['mongo_url'])
    db = client["DAP_Blinkit"]
    collection = db["AnnotationData"]
    records = list(collection.find({}))

    return pd.DataFrame(records)


# Importing dataset
reviews_df = read_collection()
reviews_df.head()

,_id,reviewId,userName,userImage,content,score,thumbsUpCount,reviewCreatedVersion,at,replyContent,repliedAt,appVersion,at_converted,repliedAt_converted,response_time
0,6a8093e50150837abb4788f3,8e0b591f-ef17-46a5-8e2c-c486dd8b0979,A Google user,https://play-lh.googleusercontent.com/EGemoI2N...,FRAUDSTERS! Do not purchase any Electronics or...,1,0,18.9.3,2026-07-21 20:48:46,"Hi there, extremely sorry for the kind of expe...",2026-07-21 21:00:26,18.9.3,2026-07-21,2026-07-21,0.0
1,6a8093e60150837abb488b09,208106ea-6883-4e8f-b823-a49b16441c05,A Google user,https://play-lh.googleusercontent.com/EGemoI2N...,thanks blinkit 🤗,5,0,17.99.1,2026-06-06 17:54:14,"Hi there, we are delighted to hear the kind wo...",2026-06-06 18:00:04,17.99.1,2026-06-06,2026-06-06,0.0
2,6a8093e50150837abb4728af,2e819a90-9f09-4246-bf07-32499ddba463,A Google user,https://play-lh.googleusercontent.com/EGemoI2N...,delivery charges jyada lete he najdik ka ho fi...,3,0,18.13.0,2026-08-09 11:06:57,"Hi Lalita, delivery charges are crucial for su...",2026-08-09 11:15:04,18.13.0,2026-08-09,2026-08-09,0.0
3,6a8093e50150837abb47be98,378719e5-21a5-42cf-ab13-77c59eba272d,A Google user,https://play-lh.googleusercontent.com/EGemoI2N...,good,5,0,17.99.1,2026-07-12 12:57:27,"Hi there, we are delighted to hear the kind wo...",2026-07-12 13:05:08,17.99.1,2026-07-12,2026-07-12,0.0
4,6a8093e50150837abb470fba,f7f78479-165f-4015-8767-3cf25cdbd411,A Google user,https://play-lh.googleusercontent.com/EGemoI2N...,best app ever use,5,0,18.15.0,2026-08-13 21:22:34,"Hi there, we are delighted to hear the kind wo...",2026-08-13 21:28:31,18.15.0,2026-08-13,2026-08-13,0.0


In [13]:
# review = """
# The delivery was very late and customer support did not respond to my messages.
# """

review = reviews_df['content'][2]

response = chat(
    model="llama3.1:8b",
    messages=[
        {
            "role": "user",
            "content": f"""
Analyze the following customer review.

Review: 
{review}

Return the sentiment as positive, negative, or neutral. 
DO NOT ADD ANY REASONING OF YOUR OWN.
"""
        }
    ],
    options={
        "temperature": 0,
        "num_ctx": 2048,
        "num_predict": 5
    }
)

print(response.message.content)

Negative


In [14]:
# Multithreading for Sentiment Polarity Annotation (max_workers = 2)  to control KV Cache
# Batching & Checkpoints for fault tolerance
# Assigning dataset IDs

In [ ]:
def annotate_senitment_polarity() -> str:
    """ 
    Python function to annotate the input reeview using LLama 3.1 8 Billion Parameter model.
    
    @args:
        input_review : Input string which represents the review body

    @returns:
        sentiment_polarity : Python string which is one of "Positive", "Negative" or "Neutral"; denoting the sentiment polarity of any input review body.
    """
    